# 🥾 Trails3D — mapping hiking trails in 3D from LiDAR
**Point Cloud Mini Course · course project · one notebook for the whole project**

| # | Stage | Status |
|---|---|---|
| 0 | Setup | ✅ |
| 1 | **Load data** (LAS/LAZ, KITTI .bin, PLY/PCD, NPY, TXT/CSV) | ✅ |
| 2 | **What data do I have?** (report + plots) | ✅ |
| 3 | **Clean** (course L2: downsample → denoise → RANSAC ground → DBSCAN) | ✅ |
| 4 | Save cleaned data | ✅ |
| 5 | Terrain model (DTM) | TODO |
| 6 | Trail detection | TODO |
| 7 | Trail analytics (length, climb, slope, difficulty) | TODO |
| 8 | Export for the phone app | TODO |

Run top to bottom. **To use your own data**, put the file in `data/raw/` and set `DATA_FILE` in section 1.

## 0. Setup
Run once. Needs Python 3.10–3.12 for Open3D (the notebook still works without Open3D: it falls back to SciPy / scikit-learn / matplotlib).

In [ ]:
# !pip install numpy scipy scikit-learn matplotlib "laspy[lazrs]" open3d
import numpy as np, time
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
plt.rcParams["figure.dpi"] = 100

try:
    import open3d as o3d
    HAS_O3D = True
except ImportError:
    HAS_O3D = False
print("Open3D available:", HAS_O3D)

RAW, PROCESSED = Path("data/raw"), Path("data/processed")
RAW.mkdir(parents=True, exist_ok=True); PROCESSED.mkdir(parents=True, exist_ok=True)

### Course LiDAR simulator
Copied from the course notebooks — gives us a street scan with ground-truth labels to test the cleaning against.

In [ ]:
# ---- Tiny LiDAR simulator (no downloads needed) -------------------------
import numpy as np

def make_scene(seed=0, n_cars=8, n_peds=5, n_poles=6):
    """Random street scene. Returns list of objects (dicts)."""
    rng = np.random.default_rng(seed)
    objs = []
    # buildings along both sides of the street
    for side in (-1, 1):
        x = -60.0
        while x < 60:
            L = rng.uniform(8, 20)
            objs.append(dict(kind="building", c=np.array([x + L / 2, side * rng.uniform(15, 18), 4.0]),
                             size=np.array([L, 4.0, 10.0]), yaw=0.0))
            x += L + rng.uniform(1, 4)
    def free(c, r):
        return all(np.hypot(*(c[:2] - o["c"][:2])) > r + 2 for o in objs if o["kind"] != "building")
    tries = 0
    while sum(o["kind"] == "car" for o in objs) < n_cars and tries < 500:
        tries += 1
        c = np.array([rng.uniform(-50, 50), rng.choice([-7.5, -4.0, 4.0, 7.5]) + rng.normal(0, .3), -0.95])
        if np.hypot(*c[:2]) < 5 or not free(c, 3): continue
        objs.append(dict(kind="car", c=c, size=np.array([rng.uniform(3.9, 4.8), rng.uniform(1.7, 1.95), rng.uniform(1.4, 1.6)]),
                         yaw=rng.normal(0, 0.08) + rng.choice([0, np.pi])))
    tries = 0
    while sum(o["kind"] == "pedestrian" for o in objs) < n_peds and tries < 500:
        tries += 1
        c = np.array([rng.uniform(-40, 40), rng.choice([-1, 1]) * rng.uniform(9.5, 11.5), -0.85])
        if not free(c, 1): continue
        objs.append(dict(kind="pedestrian", c=c, r=0.3, h=rng.uniform(1.6, 1.85)))
    for _ in range(n_poles):
        c = np.array([rng.uniform(-50, 50), rng.choice([-1, 1]) * 11.8, 1.3])
        objs.append(dict(kind="pole", c=c, r=0.12, h=6.0))
    return objs

KINDS = ["ground", "building", "car", "pedestrian", "pole"]
_REFL = dict(ground=0.15, building=0.35, car=0.6, pedestrian=0.3, pole=0.8)

def _ray_box(o, d, ob):
    c, s, yaw = ob["c"], ob["size"] / 2, ob["yaw"]
    cy, sy = np.cos(-yaw), np.sin(-yaw)
    R = np.array([[cy, -sy, 0], [sy, cy, 0], [0, 0, 1]])
    ol = R @ (o - c); dl = d @ R.T
    with np.errstate(divide="ignore", invalid="ignore"):
        t1 = (-s - ol) / dl; t2 = (s - ol) / dl
    tmin = np.nanmax(np.minimum(t1, t2), axis=1); tmax = np.nanmin(np.maximum(t1, t2), axis=1)
    hit = (tmax >= tmin) & (tmin > 0)
    return np.where(hit, tmin, np.inf)

def _ray_cyl(o, d, ob):
    c, r, h = ob["c"], ob["r"], ob["h"]
    p = o[:2] - c[:2]; dd = d[:, :2]
    a = (dd ** 2).sum(1); b = 2 * dd @ p; cc = p @ p - r * r
    disc = b * b - 4 * a * cc
    with np.errstate(invalid="ignore", divide="ignore"):
        t = (-b - np.sqrt(disc)) / (2 * a)
    z = o[2] + t * d[:, 2]
    ok = (disc > 0) & (t > 0) & (np.abs(z - c[2]) < h / 2)
    return np.where(ok, t, np.inf)

def simulate_scan(objs, pose=(0.0, 0.0, 0.0), n_beams=32, fov=(-25.0, 3.0), az_res=0.4,
                  sensor_h=1.7, max_range=80.0, noise=0.02, seed=0):
    """Ray-cast a spinning LiDAR at pose=(x, y, yaw) in the world.
    Returns pts (N,4: x,y,z,intensity) in the SENSOR frame, labels (N,), instance ids (N,), rings (N,)."""
    rng = np.random.default_rng(seed)
    px, py, pyaw = pose
    el = np.deg2rad(np.linspace(fov[0], fov[1], n_beams))
    az = np.deg2rad(np.arange(0, 360, az_res))
    E, A = np.meshgrid(el, az, indexing="ij")
    ring = np.repeat(np.arange(n_beams), len(az))
    d_s = np.stack([np.cos(E) * np.cos(A), np.cos(E) * np.sin(A), np.sin(E)], -1).reshape(-1, 3)
    cy, sy = np.cos(pyaw), np.sin(pyaw)
    Rw = np.array([[cy, -sy, 0], [sy, cy, 0], [0, 0, 1]])
    d = d_s @ Rw.T
    o = np.array([px, py, 0.0])            # world z=0 is sensor height; ground is at -sensor_h
    T = np.full(len(d), np.inf); L = np.zeros(len(d), int); I = np.full(len(d), -1)
    with np.errstate(divide="ignore"):
        tg = np.where(d[:, 2] < 0, (-sensor_h - o[2]) / d[:, 2], np.inf)
    T = tg; L[:] = 0
    for k, ob in enumerate(objs):
        t = _ray_box(o, d, ob) if "size" in ob else _ray_cyl(o, d, ob)
        m = t < T
        T[m] = t[m]; L[m] = KINDS.index(ob["kind"]); I[m] = k
    keep = T < max_range
    T = T[keep] + rng.normal(0, noise, keep.sum())
    pts = d_s[keep] * T[:, None]          # sensor frame
    lab = L[keep]
    inten = np.clip(np.array([_REFL[KINDS[l]] for l in lab]) * np.exp(-T / 120) + rng.normal(0, 0.03, len(T)), 0, 1)
    return np.c_[pts, inten].astype(np.float32), lab, I[keep], ring[keep]

def box_corners_bev(ob):
    """4 BEV corners of a box object (world frame)."""
    l, w = ob["size"][:2] / 2
    c, s = np.cos(ob["yaw"]), np.sin(ob["yaw"])
    loc = np.array([[l, w], [l, -w], [-l, -w], [-l, w]])
    return loc @ np.array([[c, s], [-s, c]]) + ob["c"][:2]
# --------------------------------------------------------------------------

### Test data generators
No real data yet? These make files in `data/raw/`:
- `make_sample_trail()` → a 100×100 m hilly trail with trees, birds/sensor noise, multipath errors and duplicates (`.laz`, like aerial LiDAR)
- `make_sample_street()` → a 64-beam course simulator scan + 800 rain/dust points (KITTI `.bin`, like L2 §3)

In [ ]:
def make_sample_street(path=RAW / "sample_street.bin"):
    pts, lab, inst, ring = simulate_scan(make_scene(seed=3, n_cars=10, n_peds=6), n_beams=64, az_res=0.2, seed=3)
    rng = np.random.default_rng(0)
    noise = rng.uniform([-40, -20, -1.5], [40, 20, 3], size=(800, 3))
    pts = np.vstack([pts, np.c_[noise, rng.uniform(0, 1, 800)]]).astype(np.float32)
    pts.tofile(path)
    np.save(path.with_suffix(".labels.npy"), np.r_[lab, np.full(800, -1)])   # ground truth, for checking
    print(f"Saved {len(pts):,} points -> {path}")
    return path


def make_sample_trail(path=RAW / "sample_trail.laz"):
    rng = np.random.default_rng(42)
    ground_z = lambda x, y: 5 * np.sin(x / 15) + 3 * np.cos(y / 12) + 0.002 * x * y
    n = 150_000
    x, y = rng.uniform(0, 100, n), rng.uniform(0, 100, n)
    on_trail = np.abs(y - (50 + 15 * np.sin(x / 20))) < 1.0          # winding 2 m trail, slightly worn
    z = ground_z(x, y) - 0.1 * on_trail + rng.normal(0, 0.03, n)
    trees = []
    for _ in range(60):
        tx, ty = rng.uniform(0, 100, 2)
        if abs(ty - (50 + 15 * np.sin(tx / 20))) < 4:
            continue
        h, m = rng.uniform(6, 14), 1500
        r = rng.uniform(0, 2.5, m) * np.sqrt(rng.uniform(0, 1, m)); a = rng.uniform(0, 2 * np.pi, m)
        trees.append(np.c_[tx + r * np.cos(a), ty + r * np.sin(a), ground_z(tx, ty) + rng.uniform(h * 0.3, h, m)])
    birds = np.c_[rng.uniform(0, 100, (800, 2)), rng.uniform(30, 80, 800)]
    multipath = np.c_[rng.uniform(0, 100, (300, 2)), rng.uniform(-40, -15, 300)]
    xyz = np.vstack([np.c_[x, y, z], *trees, birds, multipath])
    truth = np.r_[np.full(n, 2), np.full(len(xyz) - n - 1100, 5), np.full(1100, 7)]   # 2 ground, 5 tree, 7 noise
    dup = rng.choice(len(xyz), 3000)
    xyz, truth = np.vstack([xyz, xyz[dup]]), np.r_[truth, truth[dup]]
    save_cloud({"xyz": xyz, "classification": np.ones(len(xyz), np.uint8),
                "intensity": rng.integers(200, 3000, len(xyz)).astype(np.uint16)}, path)
    np.save(path.with_suffix(".labels.npy"), truth)
    return path

---
## 1. Load data
A point cloud = an **(N, D) array**: one row per laser return, `x, y, z` + extras (L1 §1).
Here a cloud is a dict: `{"xyz": (N,3), "intensity": (N,), "classification": (N,), ...}` — only `xyz` is required.

| Format | What it is (L1 §7) |
|---|---|
| `.las` / `.laz` | geospatial standard (aerial LiDAR), read with `laspy` |
| `.bin` | KITTI: raw float32 `x,y,z,intensity`, no header |
| `.ply` / `.pcd` | header + data (Open3D) |
| `.npy` / `.txt` / `.csv` | plain arrays |

In [ ]:
def _from_array(arr, fmt):
    cloud = {"xyz": arr[:, :3].astype(np.float64)}
    if arr.shape[1] >= 4:
        cloud["intensity"] = arr[:, 3]
    cloud["_meta"] = {"format": fmt, "fields": list(cloud)}
    return cloud


def load_cloud(path):
    path = Path(path); ext = path.suffix.lower()
    if ext in (".las", ".laz"):
        import laspy
        las = laspy.read(path)
        cloud = {"xyz": np.vstack([las.x, las.y, las.z]).T.astype(np.float64)}
        dims = set(las.point_format.dimension_names)
        for name in ("intensity", "classification", "return_number", "number_of_returns", "gps_time"):
            if name in dims:
                cloud[name] = np.asarray(las[name])
        if {"red", "green", "blue"} <= dims:
            cloud["rgb"] = np.vstack([las.red, las.green, las.blue]).T
        try:
            crs = las.header.parse_crs(); crs = crs.name if crs else None
        except Exception:
            crs = None
        cloud["_meta"] = {"format": f"LAS {las.header.version}", "crs": crs,
                          "fields": list(las.point_format.dimension_names)}
        return cloud
    if ext == ".bin":
        return _from_array(np.fromfile(path, dtype=np.float32).reshape(-1, 4), "KITTI bin")
    if ext == ".npy":
        return _from_array(np.load(path), "NPY")
    if ext in (".ply", ".pcd"):
        pcd = o3d.io.read_point_cloud(str(path))
        cloud = {"xyz": np.asarray(pcd.points)}
        if pcd.has_colors():
            cloud["rgb"] = (np.asarray(pcd.colors) * 255).astype(np.uint8)
        cloud["_meta"] = {"format": ext[1:].upper(), "fields": list(cloud)}
        return cloud
    if ext in (".txt", ".xyz", ".csv"):
        d = "," if ext == ".csv" else None
        try:
            arr = np.loadtxt(path, delimiter=d)
        except ValueError:
            arr = np.loadtxt(path, delimiter=d, skiprows=1)
        return _from_array(arr, "text")
    raise ValueError(f"Unsupported file type: {ext}")


def subset(cloud, mask):
    """Keep only points where mask is True — applied to every field at once."""
    out = {k: v[mask] for k, v in cloud.items() if not k.startswith("_")}
    out["_meta"] = cloud.get("_meta", {})
    return out


def save_cloud(cloud, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True); ext = path.suffix.lower()
    xyz = cloud["xyz"]
    if ext in (".las", ".laz"):
        import laspy
        header = laspy.LasHeader(point_format=3 if "rgb" in cloud else 1, version="1.2")
        header.offsets, header.scales = xyz.min(0), [0.001] * 3
        if "cluster" in cloud:
            header.add_extra_dim(laspy.ExtraBytesParams(name="cluster", type=np.int32))
        las = laspy.LasData(header)
        las.x, las.y, las.z = xyz.T
        for name in ("intensity", "classification", "return_number", "number_of_returns"):
            if name in cloud:
                las[name] = cloud[name]
        if "cluster" in cloud:
            las.cluster = cloud["cluster"].astype(np.int32)
        if "rgb" in cloud:
            las.red, las.green, las.blue = cloud["rgb"].T
        las.write(path)
    elif ext == ".bin":
        np.c_[xyz, cloud.get("intensity", np.zeros(len(xyz)))].astype(np.float32).tofile(path)
    elif ext in (".ply", ".pcd"):
        o3d.io.write_point_cloud(str(path), o3d.geometry.PointCloud(o3d.utility.Vector3dVector(xyz)))
    else:
        np.savetxt(path, xyz, fmt="%.3f")
    print(f"Saved {len(xyz):,} points -> {path}")

### 👉 Choose your data
- `None` → uses the fake trail (made automatically the first time)
- `"street"` → the course simulator scan
- or a path, e.g. `"data/raw/my_trail.laz"` — get free real data from **opentopography.org** (Find Data Map → draw a box on a trail → download LAZ)

In [ ]:
DATA_FILE = None          # None | "street" | "data/raw/<your file>"

if DATA_FILE is None:
    DATA_FILE = RAW / "sample_trail.laz"
    if not DATA_FILE.exists(): make_sample_trail(DATA_FILE)
elif DATA_FILE == "street":
    DATA_FILE = RAW / "sample_street.bin"
    if not DATA_FILE.exists(): make_sample_street(DATA_FILE)

DATA_FILE = Path(DATA_FILE)
cloud = load_cloud(DATA_FILE)
raw = cloud                                             # keep the original for before/after
truth_file = DATA_FILE.with_suffix(".labels.npy")       # only exists for generated samples
truth = np.load(truth_file) if truth_file.exists() else None
print(f"Loaded {len(cloud['xyz']):,} points from {DATA_FILE}")
print("first rows:\n", np.round(cloud["xyz"][:5], 3))

---
## 2. What data do I have?
Check before cleaning: size, fields, extent, density, duplicates, classes, outliers.

In [ ]:
LAS_CLASSES = {0: "never classified", 1: "unclassified", 2: "ground", 3: "low veg", 4: "medium veg",
               5: "high veg", 6: "building", 7: "low noise", 9: "water", 11: "road", 18: "high noise"}

def report(cloud, title="cloud"):
    xyz = cloud["xyz"]; n = len(xyz); meta = cloud.get("_meta", {})
    print(f"=== {title} ===\nPoints: {n:,}")
    for k in ("format", "crs"):
        if meta.get(k): print(f"{k}: {meta[k]}")
    print("Fields:", ", ".join(meta.get("fields", [k for k in cloud if not k.startswith('_')])))
    mn, mx = xyz.min(0), xyz.max(0); size = mx - mn
    print("\n-- Extent --")
    for i, a in enumerate("XYZ"):
        print(f"{a}: {mn[i]:9.2f} .. {mx[i]:9.2f}   (size {size[i]:.2f} m)")
    area = size[0] * size[1]
    print(f"Footprint {area:,.0f} m²  ·  density {n / area:.1f} pts/m²")
    dup = n - len(np.unique(np.round(xyz, 3), axis=0))
    q1, q99 = np.percentile(xyz[:, 2], [1, 99])
    print(f"\n-- Quality --\nDuplicates: {dup:,} ({dup / n:.1%})   NaN: {int(np.isnan(xyz).any(1).sum())}")
    print(f"Z 1%..99%: {q1:.2f} .. {q99:.2f}  → {int(((xyz[:,2] < q1) | (xyz[:,2] > q99)).sum()):,} points outside (possible noise)")
    if "classification" in cloud:
        print("\n-- Classes --")
        for v, c in zip(*np.unique(cloud["classification"], return_counts=True)):
            print(f"{int(v):>3} {LAS_CLASSES.get(int(v), 'other'):<17} {c:>9,} ({c / n:.1%})")
    if "intensity" in cloud:
        it = cloud["intensity"]; print(f"\nIntensity: min {it.min():.2f}  mean {it.mean():.2f}  max {it.max():.2f}")
    if "cluster" in cloud:
        c = cloud["cluster"]; print(f"\nClusters: {c.max() + 1}  (unclustered incl. ground: {int((c < 0).sum()):,})")

report(cloud, DATA_FILE.name)

In [ ]:
def plot_overview(cloud, color=None, title="", max_pts=300_000, cmap="viridis"):
    """Top view (BEV), side view, height histogram — like the course bev_plot."""
    xyz = cloud["xyz"]
    idx = np.random.default_rng(0).choice(len(xyz), max_pts, replace=False) if len(xyz) > max_pts else slice(None)
    c = (xyz[:, 2] if color is None else color)[idx]
    fig, ax = plt.subplots(1, 3, figsize=(18, 5), gridspec_kw={"width_ratios": [1.1, 1.6, 1]})
    ax[0].scatter(xyz[idx, 0], xyz[idx, 1], c=c, s=0.3, cmap=cmap); ax[0].set_aspect("equal"); ax[0].set_title("top view (x-y)")
    ax[1].scatter(xyz[idx, 0], xyz[idx, 2], c=c, s=0.3, cmap=cmap); ax[1].set_title("side view (x-z)")
    ax[2].hist(xyz[:, 2], bins=100, color="#4a7"); ax[2].set_yscale("log"); ax[2].set_title("height (z) histogram")
    fig.suptitle(f"{title}  —  {len(xyz):,} points"); plt.tight_layout(); plt.show()


def show_3d(cloud, color=None):
    """Interactive Open3D window (local only, not Colab). Mouse to rotate, Q to close."""
    if not HAS_O3D:
        print("Open3D not installed - use plot_overview instead"); return
    import matplotlib.cm as cm
    v = cloud["xyz"][:, 2] if color is None else np.asarray(color, float)
    lo, hi = np.percentile(v, [2, 98])
    pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(cloud["xyz"]))
    pcd.colors = o3d.utility.Vector3dVector(cm.viridis(np.clip((v - lo) / max(hi - lo, 1e-9), 0, 1))[:, :3])
    o3d.visualization.draw_geometries([pcd])

plot_overview(cloud, title="raw data, colored by height")
# show_3d(cloud)

**Density check (L1 §3):** for a spinning sensor (street scan), density falls ~1/d² — always look at results by distance.
Aerial / trail tiles are roughly uniform instead, so this is only shown for sensor-centred scans.

In [ ]:
xy = cloud["xyz"][:, :2]
sensor_centred = np.abs(np.median(xy, 0)).max() < 0.1 * np.ptp(xy, 0).max()
if sensor_centred:
    r = np.linalg.norm(xy, axis=1); bins = np.arange(0, 81, 5)
    cnt, _ = np.histogram(r, bins); area = np.pi * (bins[1:] ** 2 - bins[:-1] ** 2)
    plt.figure(figsize=(7, 3.5)); plt.semilogy(bins[:-1] + 2.5, cnt / area, "o-")
    plt.xlabel("range [m]"); plt.ylabel("points / m² (log)"); plt.title("density vs range"); plt.show()
else:
    print("Not a sensor-centred scan (aerial/trail tile) - density vs range doesn't apply.")

---
## 3. Clean the data
Course Lecture 2 pipeline, one step per cell so you can see what each does:

`hygiene (NaN, duplicates) → 3a downsample → 3b denoise → 3c ground (RANSAC) → 3d clusters (DBSCAN)`

Every step returns `(new_cloud, how_many_removed)`. All settings are in the next cell.

In [ ]:
# ---- settings (course values in brackets) ---------------------------------
VOXEL         = 0.1    # m  voxel size [L2 tried 0.05 .. 1.0]
SOR_K         = 20     # neighbours for statistical outlier removal [20]
SOR_STD       = 2.0    # remove if mean distance > mu + STD*sigma [2.0]
RANSAC_THR    = 0.15   # m  plane distance threshold [0.15]
RANSAC_ITERS  = 200    # [200]
PATCH         = 5      # m  None = one plane (flat street, course version); 5 = plane per 5x5 m patch (hills!)
MAX_TILT      = 45     # deg  ignore candidate planes steeper than this (walls, trunks)
DBSCAN_EPS    = 0.6    # m [0.6]
DBSCAN_MIN    = 5      # [5]
# ---------------------------------------------------------------------------
log = []
def step(name, result):
    global cloud
    cloud, removed = result
    log.append((name, removed, len(cloud["xyz"])))
    print(f"{name:<22} removed {removed:>9,}   left {len(cloud['xyz']):>9,}")

### Hygiene: NaNs, duplicates, noise already labelled in the file

In [ ]:
def remove_nan(c):
    keep = ~np.isnan(c["xyz"]).any(1); return subset(c, keep), int((~keep).sum())

def remove_duplicates(c, precision=0.001):
    _, idx = np.unique(np.round(c["xyz"] / precision).astype(np.int64), axis=0, return_index=True)
    keep = np.zeros(len(c["xyz"]), bool); keep[idx] = True
    return subset(c, keep), int((~keep).sum())

def remove_labelled_noise(c):
    if "classification" not in c: return c, 0
    keep = ~np.isin(c["classification"], (7, 18)); return subset(c, keep), int((~keep).sum())

cloud = raw
step("remove_nan", remove_nan(cloud))
step("remove_duplicates", remove_duplicates(cloud))
step("remove_labelled_noise", remove_labelled_noise(cloud))

### 3a. Voxel downsampling (L2 §2)
One point per cube → evens out density, makes everything after it faster.

In [ ]:
def voxel_downsample(c, voxel):
    key = np.floor((c["xyz"] - c["xyz"].min(0)) / voxel).astype(np.int64)
    _, idx = np.unique(key, axis=0, return_index=True)
    keep = np.zeros(len(key), bool); keep[idx] = True
    return subset(c, keep), int((~keep).sum())

for v in (0.05, 0.1, 0.2, 0.5, 1.0):
    print(f"voxel {v:4.2f} m → {len(voxel_downsample(cloud, v)[0]['xyz']):9,} points")
step("voxel_downsample", voxel_downsample(cloud, VOXEL))

### 3b. Statistical outlier removal (L2 §3)
For each point: mean distance to its k nearest neighbours (KD-tree, L2 §1). Far above average → noise (birds, dust, sensor errors).

In [ ]:
def statistical_outliers(c, k=20, std_ratio=2.0):
    d, _ = cKDTree(c["xyz"]).query(c["xyz"], k=k + 1)
    mean_d = d[:, 1:].mean(1)
    keep = mean_d <= mean_d.mean() + std_ratio * mean_d.std()
    return subset(c, keep), int((~keep).sum()), keep

before = cloud
new, n_removed, keep = statistical_outliers(cloud, SOR_K, SOR_STD)
step("statistical_outliers", (new, n_removed))

plt.figure(figsize=(14, 4))
plt.scatter(before["xyz"][keep, 0], before["xyz"][keep, 2], s=0.2, c="lightgray", label="kept")
plt.scatter(before["xyz"][~keep, 0], before["xyz"][~keep, 2], s=5, c="crimson", label="removed as noise")
plt.legend(markerscale=5); plt.title("side view: what the outlier filter removed"); plt.show()

### 3c. Ground with RANSAC (L2 §5)
Pick 3 points → fit a plane → count points within `RANSAC_THR` → repeat, keep the best.

⚠️ One plane only works on a **flat** street. On hills it fails (that's L2 lab A), so `PATCH = 5` fits a separate plane in every 5×5 m patch.
Two more tricks so it doesn't fit a plane through a tree: ignore steep planes (`MAX_TILT`) and pick the 3 points from the lowest half (ground is at the bottom).

Ground is **labelled** (class 2), not deleted — the terrain stage needs it.

In [ ]:
def ransac_plane(P, thr=0.15, iters=200, seed=0, max_tilt=None, low_fraction=1.0):
    rng = np.random.default_rng(seed)
    best, best_plane = np.zeros(len(P), bool), np.array([0, 0, 1.0, 0])
    if len(P) < 3: return best_plane, best
    pool = np.flatnonzero(P[:, 2] <= np.quantile(P[:, 2], low_fraction)) if low_fraction < 1 else np.arange(len(P))
    if len(pool) < 3: pool = np.arange(len(P))
    min_nz = np.cos(np.deg2rad(max_tilt)) if max_tilt is not None else 0
    for _ in range(iters):
        a, b, c = P[rng.choice(pool, 3, replace=False)]
        n = np.cross(b - a, c - a); norm = np.linalg.norm(n)
        if norm < 1e-9: continue
        n /= norm
        if abs(n[2]) < min_nz: continue
        mask = np.abs(P @ n - n @ a) < thr
        if mask.sum() > best.sum(): best, best_plane = mask, np.r_[n, -n @ a]
    return best_plane, best


def ransac_ground(c, thr=0.15, iters=200, patch=None, max_tilt=45):
    xyz = c["xyz"]; ground = np.zeros(len(xyz), bool)
    if patch:
        _, groups = np.unique(np.floor((xyz[:, :2] - xyz[:, :2].min(0)) / patch).astype(np.int64), axis=0, return_inverse=True)
        groups = groups.ravel()
    else:
        groups = np.zeros(len(xyz), int)
    for g in np.unique(groups):
        idx = np.flatnonzero(groups == g)
        _, m = ransac_plane(xyz[idx], thr, iters, seed=int(g), max_tilt=max_tilt, low_fraction=0.5 if patch else 1.0)
        ground[idx[m]] = True
    out = dict(c); out["classification"] = np.where(ground, 2, 1).astype(np.uint8)
    return out, 0

t = time.time()
step("ransac_ground", ransac_ground(cloud, RANSAC_THR, RANSAC_ITERS, PATCH, MAX_TILT))
g = cloud["classification"] == 2
print(f"ground: {g.mean():.1%} of points  ({time.time() - t:.1f} s)")
plot_overview(cloud, color=g.astype(int), title="ground (yellow) vs everything else", cmap="viridis")

### 3d. DBSCAN clusters on non-ground points (L2 §6)
Groups what's left into objects: on a trail that's trees, rocks, benches, people.

In [ ]:
def dbscan_clusters(c, eps=0.6, min_points=5):
    sel = c["classification"] != 2 if "classification" in c else np.ones(len(c["xyz"]), bool)
    labels = np.full(len(c["xyz"]), -1)
    if sel.any():
        if HAS_O3D:
            labels[sel] = np.array(o3d.geometry.PointCloud(o3d.utility.Vector3dVector(c["xyz"][sel])).cluster_dbscan(eps=eps, min_points=min_points))
        else:
            from sklearn.cluster import DBSCAN
            labels[sel] = DBSCAN(eps=eps, min_samples=min_points).fit_predict(c["xyz"][sel])
    out = dict(c); out["cluster"] = labels
    return out, 0

step("dbscan_clusters", dbscan_clusters(cloud, DBSCAN_EPS, DBSCAN_MIN))
lab = cloud["cluster"]; ng = cloud["classification"] != 2
print(f"{lab.max() + 1} clusters, {int(((lab < 0) & ng).sum()):,} non-ground points in no cluster")
plt.figure(figsize=(8, 7))
plt.scatter(cloud["xyz"][~ng, 0], cloud["xyz"][~ng, 1], s=0.1, c="#ddd")
plt.scatter(cloud["xyz"][ng, 0], cloud["xyz"][ng, 1], s=1, c=lab[ng] % 20, cmap="tab20")
plt.gca().set_aspect("equal"); plt.title("non-ground clusters (grey = ground)"); plt.show()

### Cleaning summary + accuracy check
For the generated samples we know the true labels, so we can measure how good the ground detection is (precision = how much of what we called ground really is ground; recall = how much of the real ground we found).

In [ ]:
start = len(raw["xyz"])
print(f"{'step':<22}{'removed':>10}{'left':>11}")
for name, rem, left in log: print(f"{name:<22}{rem:>10,}{left:>11,}")
print(f"\nTotal: {start:,} → {len(cloud['xyz']):,} points ({1 - len(cloud['xyz']) / start:.1%} removed)")

if truth is not None:
    # match cleaned points back to the raw points to look up their true label
    _, nearest = cKDTree(raw["xyz"]).query(cloud["xyz"])
    t = truth[nearest]
    true_ground = (t == 2) if DATA_FILE.suffix == ".laz" else (t == 0)
    pred = cloud["classification"] == 2
    print(f"\nGround precision {true_ground[pred].mean():.3f}   recall {pred[true_ground].mean():.3f}")
    noise_label = 7 if DATA_FILE.suffix == ".laz" else -1
    _, raw_nn = cKDTree(cloud["xyz"]).query(raw["xyz"][truth == noise_label])
    print(f"Noise points that survived cleaning: {(raw_nn < 1e-6).mean():.1%}")

---
## 4. Save the cleaned data
Saved as LAZ with ground (class 2) and cluster ids, plus a text report.

In [ ]:
OUT = PROCESSED / f"{DATA_FILE.stem}_clean.laz"
save_cloud(cloud, OUT)
import contextlib, io
buf = io.StringIO()
with contextlib.redirect_stdout(buf): report(cloud, OUT.name)
OUT.with_suffix(".txt").write_text(buf.getvalue(), encoding="utf-8")
report(cloud, OUT.name)

---
## 5. Terrain model (DTM) — TODO
Turn the ground points into a regular height grid.
- Grid the ground points (like the BEV lab in L1): mean z per cell, e.g. 0.5 m cells
- Fill empty cells (`scipy.interpolate.griddata`)
- Show it as an image / hillshade

Output: `dtm` (2D array), `dtm_origin`, `dtm_res`

In [ ]:
# TODO

---
## 6. Trail detection — TODO
Find the trail as an ordered 3D line.
- Easiest start: load a GPX track of the trail and snap it onto the DTM
- Harder: detect it — a narrow strip of smooth, bare ground with no vegetation above

Output: `trail` = (M, 3) points along the trail, every ~1 m

In [ ]:
# TODO

---
## 7. Trail analytics — TODO
What a hiker wants to know: length, elevation gain/loss, max slope, elevation profile, steep sections, tree cover, difficulty.
- length = sum of 3D distances between trail points
- slope = Δz / horizontal distance per segment

In [ ]:
# TODO

---
## 8. Export for the phone app — TODO
- `trail.geojson`: trail line + stats (works with any map library)
- light 3D file (`.ply`) — voxel-downsample to ~0.5 m first so phones can handle it

In [ ]:
# TODO

---
## Findings / notes
- 
- 
-